# Using agents in LlamaIndex

> An Agent is a system that leverages an AI model to interact with it's environment to achieve a user-defined objective. It combines reasoning, planning, and action execution to fulfil tasks.

LlamaIndex support three main types of reasoning agents:

1. `Function Calling Agents` - These work with AI models that can call specific functions
2. `ReAct Agents` - These work with any AI that does chat or text endpoint and deal with complex reasoning tasks.
3. `Advanced Custom Agents`  - These use more complex methods to deal withmore complex tasks and workflows.


In [32]:
# setup LM studio

from llama_index.llms.openai import OpenAI
llm = OpenAI(
    api_base = "http://localhost:1234/v1",
    api_key="lm-studio",
    model="o1", # need to trick the OpenAI class
)

In [33]:
from llama_index.core.agent.workflow import AgentWorkflow
from llama_index.core.tools import FunctionTool


def multiply(a: int, b:int) -> int:
    """Multiplies two integers and returns the resulting integer"""
    return a*b

agent = AgentWorkflow.from_tools_or_functions(
    [FunctionTool.from_defaults(multiply)],
    llm=llm
)

In [34]:
# stateless
response = await agent.run("What is 2 times 2?")
print(response,"\n")

# remembering state
from llama_index.core.workflow import Context

ctx = Context(agent) # where context store

response = await agent.run("My name is Bob.", ctx = ctx)
print(response)
response = await agent.run("What was my name again?", ctx = ctx)
print(response)


<|channel>thought
<channel|>2 times 2 is 4. 

Nice to meet you, Bob! How can I help you today?
Your name is Bob.


## Creating RAG Agents with QueryEngineTools

In [35]:
# setup vector database
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

db = chromadb.PersistentClient(path="./alfred_chroma_db")
chroma_collection = db.get_or_create_collection("alfred")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)



In [36]:
from llama_index.core.tools import QueryEngineTool
from llama_index.core import VectorStoreIndex

# setup index for llm to query
embed_model = HuggingFaceEmbedding(model_name = "BAAI/bge-small-en-v1.5")
index = VectorStoreIndex.from_vector_store(vector_store, embed_model=embed_model)

query_engine = index.as_query_engine(llm=llm, similarity_top_k=3) # as shown in the Components in LlamaIndex section

query_engine_tool = QueryEngineTool.from_defaults(
    query_engine=query_engine,
    name="name",
    description="a specific description",
    return_direct=False,
)

query_engine_agent = AgentWorkflow.from_tools_or_functions(
    [query_engine_tool],
    llm=llm,
    system_prompt="You are a helpful assistant that has access to a database containing persona descriptions."
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

### Creating Multi-agent systems

Agents in LlamaIndex can be used as tools for other agents.

In [37]:
from llama_index.core.agent.workflow import (
    AgentWorkflow,
    FunctionAgent,
    ReActAgent,
)


def add(a:int,b:int) -> int:
    """Add two numbers."""
    return a+b

def subtract(a:int, b:int) -> int:
    """Subtract two numbers."""
    return a-b

# we use FunctionAgent or ReActAgent here
# FunctionAgent works for LLMs with a function calling API
# ReActAgent works for any LLM

calculator_agent = ReActAgent(
    name = "calculator",
    description = "Performs basic arithmetic operations",
    system_prompt = "You are a calculator assistant. use your tools for any math operation.",
    tools = [add,subtract],
    llm = llm,
)

query_agent = ReActAgent(
    name = "info_lookup",
    description = "Looks up information about XYZ",
    system_prompt = "Use your tool to query a RAG system to answer information about XYZ",
    tools = [query_engine_tool],
    llm=llm
)

agent = AgentWorkflow(
    agents = [calculator_agent, query_agent], root_agent = "calculator"
)

response = await agent.run(user_msg = "Can you add 5 and 3?")
print(response)





5 + 3 = 8
